In [3]:
!pip install -q llama-index-core llama-index-embeddings-huggingface llama-index-vector-stores-chroma sentence-transformers chromadb pypdf request

ERROR: Could not find a version that satisfies the requirement request (from versions: none)
ERROR: No matching distribution found for request


In [17]:
PDF_DIR  = "/content"
CHROMA_DB_PATH = "./chroma_db/rag_docs"
COLLECTION_NAME = "rag_docs"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 200
TOP_K = 10
API_KEY = "your-openrouter-api-key"
MODEL_NAME = "openai/gpt-4o-mini"
MAX_TOKENS = 4096

In [7]:
from pathlib import Path
from pypdf import PdfReader
from llama_index.core import Document

pdf_path = Path(PDF_DIR)
pdf_files = list(pdf_path.glob("*.pdf"))
raw_document = []
for pdf_file in pdf_files:
    try:
        reader = PdfReader(str(pdf_file))
        total_pages = len(reader.pages)

        print(f"{pdf_file.name} - {total_pages}")

        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if not text.strip():
                continue
            raw_document.append(Document(
                text = text,
                metadata = {
                    "source":pdf_file.name,
                    "page_num": page_num +1,
                    "total_pages": total_pages
                }
            ))
    except Exception as E:
        print("Document cannot be loaded due to", E)

EventForce_Management_System_Naan_Mudhalvan_Report.pdf - 24


In [8]:
raw_document[0]

Document(id_='b92b4b4b-9f44-4ceb-a0ee-11ae2d703882', embedding=None, metadata={'source': 'EventForce_Management_System_Naan_Mudhalvan_Report.pdf', 'page_num': 1, 'total_pages': 24}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='EVENTFORCE MANAGEMENT SYSTEM\nSalesforce-Based Event Planning, Venue Reservation and Automated Vendor Coordination\nPlatform Using Agentforce and Flow Builder\nProject By: BALAJI B\nCollege: NADAR SARASWATHI COLLEGE OF ENGINEERING AND TECHNOLOGY.\nDepartment: B.Tech ARTIFICIAL INTELLIGENCE & DATA SCIENCE\nCollege Code: 9210\nNM ID: bae6786609a4cdfbc7641f833428fe63\nEmail: yuvabalaji572@gmail.com\nTeam ID :\nTeam Size : 4\nTeam Leader : Balaji B\nTeam member : Ajay Prasath K    Team member : Devendra Kumar P    Team member : Gokul M\n', path=None, url=None, mimetype=None), image_resource=None, audio_resourc

In [9]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP
)

raw_chunk = splitter.get_nodes_from_documents(raw_document)

In [11]:
raw_chunk[11]

TextNode(id_='e9c0f6d4-4c84-4af9-b028-1b8d667c2929', embedding=None, metadata={'source': 'EventForce_Management_System_Naan_Mudhalvan_Report.pdf', 'page_num': 12, 'total_pages': 24}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='aaecba74-e765-4084-bea9-34d09c275ecf', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'source': 'EventForce_Management_System_Naan_Mudhalvan_Report.pdf', 'page_num': 12, 'total_pages': 24}, hash='9645f5074475f3bd857217e4547a87337a822d46fa43a10cad2232ae264f6f9d')}, metadata_template='{key}: {value}', metadata_separator='\n', text="PRIORITY PREDICTION AND ASSIGNMENT LOGIC\nPriority Levels\n Priority\nExample Operational Indicators\nOperational Handling\n High\nAudience > 500, VIP keynote, budget > $50,000, or booking\nlead time < 7 days\nExpedited routing, senior planner assignment, instant\nmanager SMS alert\n Medium\nAudience 100-500, standard corporate meeting, budget\n$

In [12]:
def clean_chunk(text):
    text = str(text.encode('utf-8', errors = 'ignore').decode('utf-8'))
    return "".join(text.strip())

chunk = []
for node in raw_chunk:
    content = node.get_content().strip()
    if content:
        node.text = clean_chunk(content)
        chunk.append(node)

In [13]:
chunk[18]

TextNode(id_='b01640f8-2b59-40d5-bd2e-195313240d2e', embedding=None, metadata={'source': 'EventForce_Management_System_Naan_Mudhalvan_Report.pdf', 'page_num': 19, 'total_pages': 24}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='e1a23de3-062a-467b-ab0e-f5f9318c01c7', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'source': 'EventForce_Management_System_Naan_Mudhalvan_Report.pdf', 'page_num': 19, 'total_pages': 24}, hash='4b0d564e74a0217b51ffaf76626a86b2f3106536d5d685364cbbe44d0718b9ea')}, metadata_template='{key}: {value}', metadata_separator='\n', text='REPORTS, DASHBOARDS AND USER INTERFACE (CONTD.)\nData Quality & Visual Reporting\nVisual reports require consistent timestamps, standard picklists, and reliable field history tracking. The dashboard components\nbelow showcase live operational telemetry generated within the EventForce Lightning App environment.\n Figure 15.1: Salesforce Lightning 

In [14]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embedding = HuggingFaceEmbedding(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

print("model_loaded_sucessfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model_loaded_sucessfully


In [15]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

chroma_db = Path(CHROMA_DB_PATH)

chroma_client = chromadb.PersistentClient(path = chroma_db)
chroma_collection = chroma_client.get_or_create_collection(COLLECTION_NAME)
vector_store = ChromaVectorStore(chroma_collection = chroma_collection)

index = VectorStoreIndex.from_vector_store(
    vector_store = vector_store,
    embed_model = embedding
)
if not chunk:
    print("No chunks to loaded into vector_db")
index.insert_nodes(chunk)
print(chroma_collection.count())

24


In [18]:
import requests

class OpenRouterLLM:

    def __init__(self, api_key: str, model: str =MODEL_NAME, max_token: int = MAX_TOKENS ):
        self.api_key  = api_key
        self.model = model
        self.max_token = max_token

    def invoke(self, prompt):

        response = requests.post(
            url = "https://openrouter.ai/api/v1/chat/completions",
            headers = {
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            },
            json = {
                "model": self.model,
                "max_tokens": self.max_token,
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            },
            timeout = 60
        )

        response.raise_for_status()

        data = response.json()

        return data["choices"][0]["message"]["content"]

LLM = OpenRouterLLM(api_key = API_KEY)
LLM.invoke("Who is the president of india")


'As of my last knowledge update in October 2023, the President of India is Droupadi Murmu. She took office on July 25, 2022, becoming the first tribal woman to hold the position. Please verify with the latest sources, as my information may be outdated.'

In [19]:
Prompt_Template = """
You are an expert study assistant. Using ONLY the context provided below,
give a thorough and detailed answer to the question.

Your answer should:
- Be comprehensive and well-structured
- Use bullet points or numbered lists where appropriate
- Include relevant examples or explanations from the context
- Use headings to organise if the answer covers multiple topics
- Be as detailed as possible — do NOT summarise briefly
- If the answer is not in the context, say "I don't have enough information."

context : {context},

Question :{question}

Detailed Answer

"""

chroma_retriever = index.as_retriever(
    similiarity_top_k = TOP_K
)

def format_nodes(nodes):
    parts = []
    for n in nodes:
        source = n.node.metadata.get('source','unknown')
        page = n.node.metadata.get('page_num', 'N/A')
        parts.append(f'[Source: {source} | Page {page}]\n {n.node.get_content()}')

        return "\n\n".join(parts)
print("Rag Pipeline is ready")

Rag Pipeline is ready


In [21]:
def ask(question: str):

    print(f"\n{'='*60}")
    print(f' Question: {question}')
    print('='*60)

    nodes = chroma_retriever.retrieve(question)
    content_str =  format_nodes(nodes)
    prompt = Prompt_Template.format(
        context = content_str,
        question = question
    )

    try:
        answer = LLM.invoke(prompt)
    except Exception as E:
        print("Error:", E)
        return ''

    print("Anwser: ", answer)

ask('Explain about Document')


 Question: Explain about Document
Anwser:  # Detailed Explanation of the Document

The document discusses various functionalities and actions within the Agentforce system, particularly focusing on its capabilities for managing event-related tasks efficiently. Below, the key components of the document are detailed to provide a comprehensive understanding of the features and operational mechanisms of Agentforce.

## Suggested Agent Actions

The document lists several specific actions that the Agentforce system can perform, each with defined input parameters and output values. Here’s a detailed breakdown of these actions:

### 1. Check Venue Availability
- **Input Parameters:**
  - `venueId`: Identifier for the specific venue being checked.
  - `eventDate`: Date when the event is expected to occur.
  - `startTime`: Starting time of the event.
  - `endTime`: Ending time of the event.
- **Output Values:**
  - `isAvailable (Boolean)`: Indicates whether the venue is available for booking.
  